# LAB 08 — TOKENIZATION VÀ MASKED LANGUAGE MODELING (BERT)

**Môn học**: Xử lý ngôn ngữ tự nhiên và ứng dụng  
**Học kỳ**: Học kỳ I - 2026  
**Sinh viên thực hiện**: Nguyễn Văn Thành  
**Mục tiêu**: Khảo sát cơ chế WordPiece Tokenization, Special Tokens, Attention Mask và suy luận Masked Language Modeling với BERT.

## 1. Nạp Thư Viện và Khởi Tạo Tokenizer
Sử dụng `BertTokenizer` từ `transformers` với bộ từ vựng chuẩn `bert-base-uncased` ($V = 30,522$).

In [1]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print('Vocab size:', tokenizer.vocab_size)

C:\Users\Thanh\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vocab size: 30522


## 2. Calculation Exercise 2 — Phân Tích Câu 'The cats are playing.'
Thực hiện tokenization, lấy token IDs, xác định special tokens, attention mask và so sánh words vs tokens vs subwords.

In [2]:
raw_text = 'The cats are playing.'
encoded = tokenizer(raw_text, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
token_ids = encoded['input_ids'][0].tolist()
attention_mask = encoded['attention_mask'][0].tolist()

print('Raw Text:', raw_text)
print('Tokens:', tokens)
print('Token IDs:', token_ids)
print('Attention Mask:', attention_mask)
print('Special Tokens:', [t for t in tokens if t in tokenizer.all_special_tokens])
print('Word count:', len(raw_text.replace('.', '').split()))
print('Token count:', len(tokens))
print('Subword tokens (excluding specials):', tokens[1:-1])

Raw Text: The cats are playing.
Tokens: ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']
Token IDs: [101, 1996, 8870, 2024, 2652, 1012, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1]
Special Tokens: ['[CLS]', '[SEP]']
Word count: 4
Token count: 7
Subword tokens (excluding specials): ['the', 'cats', 'are', 'playing', '.']


## 3. Khảo Sát WordPiece Subword Tokenization
Khảo sát các từ ghép, tiền tố, hậu tố và từ hiếm để thấy cách WordPiece tách thành các subword mang tiền tố `##`.

In [3]:
sample_words = ['unbelievable', 'transformer', 'antigravity', 'hyperparameter', 'tokenization']
for word in sample_words:
    sub_tokens = tokenizer.tokenize(word)
    print(f'{word:16} -> {sub_tokens}')

unbelievable     -> ['unbelievable']
transformer      -> ['transform', '##er']
antigravity      -> ['anti', '##gra', '##vity']
hyperparameter   -> ['hyper', '##para', '##meter']
tokenization     -> ['token', '##ization']


## 4. Danh Sách Special Tokens Trong BERT
Kiểm tra các token đặc biệt cốt lõi: `[PAD]`, `[UNK]`, `[CLS]`, `[SEP]`, `[MASK]` cùng các ID định danh tương ứng.

In [4]:
special_tokens_list = ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
for token in special_tokens_list:
    tid = tokenizer.convert_tokens_to_ids(token)
    print(f'Token: {token:8} | ID: {tid}')

Token: [PAD]    | ID: 0
Token: [UNK]    | ID: 100
Token: [CLS]    | ID: 101
Token: [SEP]    | ID: 102
Token: [MASK]   | ID: 103


## 5. Tokenization Cặp Câu và Phân Tách Bằng Token Type IDs
BERT sử dụng `token_type_ids` (0 cho Sentence A, 1 cho Sentence B) để phân định hai phân đoạn trong bài toán Next Sentence Prediction hoặc QA.

In [5]:
sent_a = 'The cat sat on the mat.'
sent_b = 'It looked at the fish.'
pair_encoded = tokenizer(sent_a, sent_b, return_tensors='pt')
pair_tokens = tokenizer.convert_ids_to_tokens(pair_encoded['input_ids'][0])
print('Pair Tokens:', pair_tokens)
print('Token Type IDs:', pair_encoded['token_type_ids'][0].tolist())
print('Attention Mask:', pair_encoded['attention_mask'][0].tolist())

Pair Tokens: ['[CLS]', 'the', 'cat', 'sat', 'on', 'the', 'mat', '.', '[SEP]', 'it', 'looked', 'at', 'the', 'fish', '.', '[SEP]']
Token Type IDs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 6. Padding và Vai Trò Của Attention Mask
Khi gom batch các câu có độ dài khác nhau, padding tokens được chèn vào và `attention_mask` gán giá trị 0 để loại trừ các vị trí pad.

In [6]:
batch_sentences = ['Hello world', 'BERT pretrained transformers are powerful models.']
batch_encoded = tokenizer(batch_sentences, padding=True, truncation=True, return_tensors='pt')
print('Batch Keys:', list(batch_encoded.keys()))
print('Input IDs shape:', batch_encoded['input_ids'].shape)
print('Input IDs:\n', batch_encoded['input_ids'])
print('Attention Mask:\n', batch_encoded['attention_mask'])

Batch Keys: ['input_ids', 'token_type_ids', 'attention_mask']
Input IDs shape: torch.Size([2, 11])
Input IDs:
 tensor([[  101,  7592,  2088,   102,     0,     0,     0,     0,     0,     0,
             0],
        [  101, 14324,  3653, 23654,  2098, 19081,  2024,  3928,  4275,  1012,
           102]])
Attention Mask:
 tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


## 7. Masked Language Modeling (MLM) Inference
Sử dụng `BertForMaskedLM` để dự đoán từ bị che trong câu: `The cat [MASK] fish.` và lấy top 5 từ có xác suất cao nhất.

In [7]:
mlm_model = BertForMaskedLM.from_pretrained('bert-base-uncased')
mlm_model.eval()

masked_sentence = 'The cat [MASK] fish.'
inputs = tokenizer(masked_sentence, return_tensors='pt')
mask_token_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]

with torch.no_grad():
    outputs = mlm_model(**inputs)
    logits = outputs.logits

mask_logits = logits[0, mask_token_index, :]
probs = torch.softmax(mask_logits, dim=-1)[0]
topk_probs, topk_indices = torch.topk(probs, k=5)

for p, idx in zip(topk_probs, topk_indices):
    tok = tokenizer.convert_ids_to_tokens([idx.item()])[0]
    print(f'Token: {tok:10} | Probability: {p.item():.4f}')

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:  14%|█▍        | 28/202 [00:00<00:00, 278.07it/s]

Loading weights:  28%|██▊       | 56/202 [00:00<00:00, 226.39it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 630.93it/s]


[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Token: and        | Probability: 0.5909
Token: -          | Probability: 0.0706
Token: .          | Probability: 0.0401
Token: or         | Probability: 0.0209
Token: is         | Probability: 0.0198


## 8. Calculation Exercise 1 — Xác Thực Công Thức Cross-Entropy Bằng Code
Kiểm chứng tính toán $L = -\ln(P)$ với các giá trị giả thiết $P \in \{0.70, 0.95, 0.01\}$.

In [8]:
import numpy as np

p_values = [0.70, 0.95, 0.01]
for p in p_values:
    loss = -np.log(p)
    print(f'P(eats) = {p:.2f} -> Loss = -ln({p:.2f}) = {loss:.5f}')

P(eats) = 0.70 -> Loss = -ln(0.70) = 0.35667
P(eats) = 0.95 -> Loss = -ln(0.95) = 0.05129
P(eats) = 0.01 -> Loss = -ln(0.01) = 4.60517
